In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#mounting to the google drive

Mounted at /content/drive


In [ ]:
import os
WORKDIR = "/content/drive/MyDrive/URI"
os.makedirs(WORKDIR, exist_ok=True)
%cd $WORKDIR

#directory is now URI

/content/drive/MyDrive/URI


In [ ]:
!pip -q install "transformers>=4.44.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
  "peft>=0.12.0" "trl>=0.9.6" bitsandbytes "scikit-learn>=1.4.0" \
  "evaluate>=0.4.1" "huggingface_hub>=0.24.0"

  #installing the libaries needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

#see GPU stats

Sat May 23 19:06:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from huggingface_hub import login
login("token here

In [ ]:
import pandas as pd

# ---- Paths / columns (match your CSV) ----
CSV_PATH = "/content/drive/MyDrive/URI/Training.dataset.health.csv"

COL_TEXT  = "text"
COL_LABEL = "Compliant"
COL_GROUP = "source_pdf"   # used later to split by document (prevents leakage)

# ---- Load ----
df = pd.read_csv(CSV_PATH, encoding="latin1")

# ---- Validate columns ----
required = {COL_TEXT, COL_LABEL, COL_GROUP}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

# ---- Clean text ----
df[COL_TEXT] = (
    df[COL_TEXT]
    .astype(str)
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)  # collapse weird whitespace
    .str.strip()
)

# ---- Clean labels: strict 0/1 ints (fails loudly if inconsistencies exist) ----
df[COL_LABEL] = pd.to_numeric(df[COL_LABEL], errors="raise").astype(int)
bad = set(df[COL_LABEL].unique()) - {0, 1}
if bad:
    raise ValueError(f"Labels must be 0/1 only. Found: {sorted(bad)}")

# ---- Drop empty text rows ----
df = df[df[COL_TEXT].str.len() > 0].reset_index(drop=True)

# ---- Quick sanity prints ----
print("Rows:", len(df))
print("Label counts:", df[COL_LABEL].value_counts().to_dict())
print("Unique docs (source_pdf):", df[COL_GROUP].nunique())

df.head(10)





Rows: 602
Label counts: {1: 398, 0: 204}
Unique docs (source_pdf): 7


,source_pdf,page,sent_idx,text,Compliant
0,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,1,LOS ANGELES COUNTY HEALTH AGENCY LOS ANGELES C...,1
1,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,2,THIS INFORMATION.,0
2,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,3,"Effective date: May 30, 2017 WHO WILL FOLLOW T...",1
3,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,4,The Joint Notice describes about you.,0
4,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,5,It also information.,0
5,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,6,California (USC).,0
6,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,7,"Workforce case managers, students of Health Se...",0
7,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,8,This record is This of the care provided to yo...,0
8,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,9,"To manage your care, we OUR PLEDGE REGARDING Y...",1
9,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,10,HA-1050 (rev 5/2023),0


In [ ]:
import pandas as pd

# Keep only the columns you’ll use
kept_cols = ["source_pdf", "page", "sent_idx", COL_TEXT, COL_LABEL]
df = df[kept_cols].copy()

# Clean text
df[COL_TEXT] = (
    df[COL_TEXT]
    .astype(str)
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# STRICT label conversion (fail fast if any inconsistency exists)
df[COL_LABEL] = pd.to_numeric(df[COL_LABEL], errors="raise").astype(int)
bad = set(df[COL_LABEL].unique()) - {0, 1}
if bad:
    raise ValueError(f"Labels must be 0/1 only. Found: {sorted(bad)}")

# Remove empty text rows
df = df[df[COL_TEXT].str.len() > 0].reset_index(drop=True)

# OPTIONAL: check duplicate text (don’t drop yet—just inspect)
dup_count = df.duplicated(subset=[COL_TEXT]).sum()
print("Duplicate text rows:", dup_count)

# If you *really* want to dedupe, do it intentionally:
# df = df.drop_duplicates(subset=[COL_TEXT]).reset_index(drop=True)

print("Total rows:", len(df))
print("Class balance (fraction of 1s):", float(df[COL_LABEL].mean()))

df.head(5)



Duplicate text rows: 4
Total rows: 602
Class balance (fraction of 1s): 0.6611295681063123


,source_pdf,page,sent_idx,text,Compliant
0,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,1,LOS ANGELES COUNTY HEALTH AGENCY LOS ANGELES C...,1
1,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,2,THIS INFORMATION.,0
2,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,3,"Effective date: May 30, 2017 WHO WILL FOLLOW T...",1
3,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,4,The Joint Notice describes about you.,0
4,1061387_HA10502017FinalHAJointNPPHandoutEnglis...,1,5,It also information.,0


In [ ]:
import pandas as pd, json, os
from sklearn.model_selection import train_test_split

CSV_PATH = "/content/drive/MyDrive/URI/Training.dataset.health.csv"
COL_TEXT  = "text"
COL_LABEL = "Compliant"
COL_GROUP = "source_pdf"   # group key to prevent leakage

# ---- Load ----
df = pd.read_csv(CSV_PATH, encoding="latin1")

# ---- Keep columns ----
kept = ["source_pdf", "page", "sent_idx", COL_TEXT, COL_LABEL]
df = df[kept].copy()

# ---- Clean text ----
df[COL_TEXT] = (
    df[COL_TEXT]
    .astype(str)
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ---- STRICT labels (fail fast) ----
df[COL_LABEL] = pd.to_numeric(df[COL_LABEL], errors="raise").astype(int)
bad = set(df[COL_LABEL].unique()) - {0, 1}
if bad:
    raise ValueError(f"Labels must be 0/1 only. Found: {sorted(bad)}")

# ---- Remove empty rows ----
df = df[df[COL_TEXT].str.len() > 0].reset_index(drop=True)

print("Total rows:", len(df))
print("Class balance (fraction of 1s):", float(df[COL_LABEL].mean()))
print("Unique documents:", df[COL_GROUP].nunique())

# -----------------------------
# Split by DOCUMENT (leak-proof)
# -----------------------------
docs = df[COL_GROUP].dropna().unique().tolist()
train_docs, test_docs = train_test_split(
    docs,
    test_size=0.20,
    random_state=42
)

train_df = df[df[COL_GROUP].isin(train_docs)].reset_index(drop=True)
test_df  = df[df[COL_GROUP].isin(test_docs)].reset_index(drop=True)

print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Train balance:", float(train_df[COL_LABEL].mean()))
print("Test balance:", float(test_df[COL_LABEL].mean()))

# -----------------------------
# Convert to JSONL
# -----------------------------
RULE = (
    "Label the sentence using this rule:\n"
    "1 = clearly HIPAA / Notice of Privacy Practices language AND describes an allowed/rightful practice.\n"
    "0 = everything else (not HIPAA, a violation, too vague, or needs context).\n"
    "Output only 0 or 1."
)

def to_ex(r):
    lab = int(r[COL_LABEL])
    sent = r[COL_TEXT]
    prompt = f"{RULE}\n\nSentence: {sent}\nAnswer:"
    return {
        "prompt": prompt,
        "response": str(lab),
        "meta": {
            "source_pdf": r.get("source_pdf", ""),
            "page": int(r["page"]) if pd.notna(r.get("page")) else -1,
            "sent_idx": int(r["sent_idx"]) if pd.notna(r.get("sent_idx")) else -1
        }
    }

def dump_jsonl(df_, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, r in df_.iterrows():
            f.write(json.dumps(to_ex(r), ensure_ascii=False) + "\n")

os.makedirs("data", exist_ok=True)
dump_jsonl(train_df, "data/train.jsonl")
dump_jsonl(test_df,  "data/test.jsonl")

!ls -lh data/*.jsonl


Total rows: 602
Class balance (fraction of 1s): 0.6611295681063123
Unique documents: 7
Train rows: 479 Test rows: 123
Train balance: 0.6722338204592901
Test balance: 0.6178861788617886
-rw------- 1 root root  66K May 23 19:06 data/test.jsonl
-rw------- 1 root root 268K May 23 19:06 data/train.jsonl


In [ ]:
import os, torch, re
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# -------- 4-bit quant (QLoRA) --------
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

import os, torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# -------- 4-bit quant (QLoRA) --------
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
base.config.use_cache = False

base = prepare_model_for_kbit_training(base)

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

# -------- Load JSONL --------
ds = load_dataset("json", data_files={"train":"data/train.jsonl", "test":"data/test.jsonl"})

SYSTEM_RUBRIC = (
    "You are a strict classifier. Output only a single character: 0 or 1.\n"
    "Rule:\n"
    "1 = clearly HIPAA / Notice of Privacy Practices language AND describes an allowed/rightful practice.\n"
    "0 = everything else (not HIPAA, a violation, too vague, or needs context).\n"
    "If unsure, output 0."
)

MAX_LENGTH = 256

def preprocess(example):
    prompt = (
        f"<|system|>\n{SYSTEM_RUBRIC}\n"
        f"<|user|>\n{example['prompt']}\n"
        f"<|assistant|>\n"
    )
    answer = str(example["response"]).strip()
    if answer not in {"0", "1"}:
        raise ValueError(f"Bad response label: {answer}")

    full_text = prompt + answer + tok.eos_token

    enc = tok(full_text, truncation=True, max_length=MAX_LENGTH, padding=False)

    prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + enc["input_ids"][len(prompt_ids):]
    labels = labels[:len(enc["input_ids"])]

    enc["labels"] = labels
    return enc

train_tok = ds["train"].map(preprocess, remove_columns=ds["train"].column_names)
test_tok  = ds["test"].map(preprocess,  remove_columns=ds["test"].column_names)

# -------- Collator that pads labels with -100 --------
def collate_fn(features):
    # Pad input_ids/attention_mask
    batch = tok.pad(
        [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
        padding=True,
        return_tensors="pt",
    )

    # Pad labels to same length with -100
    max_len = batch["input_ids"].shape[1]
    labels = []
    for f in features:
        lab = f["labels"]
        if len(lab) < max_len:
            lab = lab + [-100] * (max_len - len(lab))
        else:
            lab = lab[:max_len]
        labels.append(lab)

    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch

# -------- Training args --------
args = TrainingArguments(
    output_dir="ckpt_llama32_1b_hipaa_lora",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    gradient_checkpointing=True,
    report_to=[],
    fp16=torch.cuda.is_available(),
    bf16=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=collate_fn,   # <-- use our safe collator
)

trainer.train()

model.save_pretrained("ckpt_llama32_1b_hipaa_lora")
tok.save_pretrained("ckpt_llama32_1b_hipaa_lora")
print("Training complete.")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/479 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


Training complete.


In [ ]:
import os, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID  = "meta-llama/Llama-3.2-1B-Instruct"
CKPT_DIR = "ckpt_llama32_1b_hipaa_lora"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base, CKPT_DIR).eval()

SYSTEM_RUBRIC = (
    "You are a strict classifier. Output only a single character: 0 or 1.\n"
    "Rule:\n"
    "1 = clearly HIPAA / Notice of Privacy Practices language AND describes an allowed/rightful practice.\n"
    "0 = everything else (not HIPAA, a violation, too vague, or needs context).\n"
    "If unsure, output 0."
)

def classify_one(sentence: str) -> int:
    prompt = (
        f"<|system|>\n{SYSTEM_RUBRIC}\n"
        f"<|user|>\nSentence: {sentence}\nAnswer:\n"
        f"<|assistant|>\n"
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=2,          # we only need 1 token
            do_sample=False,
            temperature=0.0,
            pad_token_id=tok.eos_token_id
        )

    decoded = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # pick first 0/1 found
    m = re.search(r"[01]", decoded)
    return int(m.group(0)) if m else 0




Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
import json
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

y_true = []
y_pred = []

with open("data/test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)

        # Ground truth label
        actual = int(ex["response"])

        # IMPORTANT: extract the raw sentence from the prompt
        # Prompt format: "... Sentence: {sentence}\nAnswer:"
        prompt_text = ex["prompt"]
        if "Sentence:" in prompt_text:
            sentence = prompt_text.split("Sentence:", 1)[1].split("\n", 1)[0].strip()
        else:
            raise ValueError("Could not extract sentence from prompt")

        # Model prediction (returns int 0/1)
        pred = classify_one(sentence)

        y_true.append(actual)
        y_pred.append(pred)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F1:        {f1:.4f}")
print("Confusion matrix [[tn, fp], [fn, tp]]:")
print(cm)



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Accuracy:  0.6667
Precision: 0.9268
Recall:    0.5000
F1:        0.6496
Confusion matrix [[tn, fp], [fn, tp]]:
[[44  3]
 [38 38]]


In [ ]:
df["Compliant"].value_counts(normalize=True) * 100


,proportion
Compliant,
1,66.112957
0,33.887043


In [ ]:
#STOPPPPPP!!!!! SEE ACCURACY!!!

In [ ]:
import pandas as pd, os

UNLABELED_CSV = "/content/drive/MyDrive/llama31_hipaa/SmallSample_Health_notLabeled.csv"  # <- change if needed
TEXT_COL = "text"
OUT_CSV  = "SmallSamplePredicted.csv"

df = pd.read_csv(UNLABELED_CSV)
labels, reasons = [], []
for s in df[TEXT_COL].astype(str):
    res = classify_one(s)
    labels.append(res.get("label",0))
    reasons.append(res.get("reason",""))
df["pred_label"]  = labels
df["pred_reason"] = reasons
df.to_csv(OUT_CSV, index=False)
print("wrote", os.path.abspath(OUT_CSV))


In [ ]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# (re)set your token if needed
# os.environ["HF_TOKEN"] = "hf_...your_token..."

HF_TOKEN = os.environ.get("HF_TOKEN")
BASE_ID  = "meta-llama/Llama-3.2-1B-Instruct"
CKPT_DIR = "ckpt_llama32_1b_hipaa_lora"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    quantization_config=bnb,
    device_map={"": "cuda:0"},
    low_cpu_mem_usage=True
)
model = PeftModel.from_pretrained(base, CKPT_DIR).eval()

SYSTEM = (
    'You are a HIPAA compliance screening assistant. '
    'Return JSON: {"label":0 or 1,"reason":"short"}. When unsure choose 0.'
)

def classify_one(sentence: str):
    # Build prompt and run deterministic generation (no sampling args)
    prompt = (f"<s>[SYSTEM]\n{SYSTEM}\n[/SYSTEM]\n"
              f"[USER]\nSentence: {sentence}\nDecide now.\n[/USER]\n[ASSISTANT]\n")
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    text = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    try:
        return json.loads(text.split("</s>")[0])
    except Exception:
        return {"label": 0, "reason": "malformed"}


In [ ]:
#Stopped here. Accuracy is 95%

In [ ]:
# If not already installed:
!pip -q install tqdm

import os, json, math
import pandas as pd
import torch
from tqdm import tqdm

# Speed knobs you can tweak
INPUT_CSV   = "/content/drive/MyDrive/llama31_hipaa/SmallSample_Health_notLabeled.csv"  # your file
TEXT_COL    = "text"
OUTPUT_CSV  = "/content/drive/MyDrive/llama31_hipaa/SmallSamplePredicted.csv"   # save straight to Drive
BATCH_SIZE  = 16         # try 8, 16, or 32 depending on GPU memory
MAX_NEW_TOK = 48         # lower = faster; 32–64 is plenty for short JSON
SAVE_EVERY  = 500        # autosave every N rows

# Optional minor speedups
torch.set_grad_enabled(False)
try:
    torch.set_float32_matmul_precision("high")  # PyTorch 2.x
except Exception:
    pass

# Load CSV
df = pd.read_csv(INPUT_CSV)
if TEXT_COL not in df.columns:
    # try to auto-detect a single text-like column
    possible = [c for c in df.columns if df[c].dtype == "object"]
    if len(possible) == 1:
        TEXT_COL = possible[0]
    else:
        raise ValueError(f"Couldn't find '{TEXT_COL}' column. Found: {list(df.columns)}")

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

# Batched classify (uses the classify_one prompt format, but in batches)
SYSTEM = (
    'You are a HIPAA compliance screening assistant. '
    'Return strictly valid JSON: {"label":0 or 1,"reason":"short"}. When unsure choose 0.'
)

def build_prompt(s):
    return (f"<s>[SYSTEM]\n{SYSTEM}\n[/SYSTEM]\n"
            f"[USER]\nSentence: {s}\nDecide now.\n[/USER]\n[ASSISTANT]\n")

def classify_batch(texts):
    prompts = [build_prompt(s) for s in texts]
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOK,
            do_sample=False  # deterministic
        )
    # slice off the prompt part for each sequence
    dec = []
    for i in range(len(texts)):
        gen_ids = out[i, enc["input_ids"].shape[1]:]
        text = tok.decode(gen_ids, skip_special_tokens=True).strip()
        try:
            dec.append(json.loads(text.split("</s>")[0]))
        except Exception:
            dec.append({"label": 0, "reason": "malformed"})
    return dec

pred_label, pred_reason = [], []
n = len(df)

def normalize_result(r):
    """
    Normalizes model output:
    - if r is a dict → return it
    - if r is a list → return first element if it's a dict
    - if r is malformed → return default non-compliant
    """
    if isinstance(r, dict):
        return r
    if isinstance(r, list) and len(r) > 0:
        if isinstance(r[0], dict):
            return r[0]
        else:
            return {"label": 0, "reason": "malformed-list"}
    return {"label": 0, "reason": "malformed"}

from tqdm import tqdm

for start in tqdm(range(0, n, BATCH_SIZE), desc="Labeling"):
    chunk = df[TEXT_COL].iloc[start:start+BATCH_SIZE].tolist()
    results = classify_batch(chunk)

    for r in results:
        r_norm = normalize_result(r)
        pred_label.append(int(r_norm.get("label", 0)))
        pred_reason.append(str(r_norm.get("reason", "")))

    # autosave every ~500 rows
    if (start // BATCH_SIZE) % max(1, SAVE_EVERY // max(1, BATCH_SIZE)) == 0 and start > 0:
        tmp = df.loc[:len(pred_label)-1, [TEXT_COL]].copy()
        tmp["pred_label"]  = pred_label
        tmp["pred_reason"] = pred_reason
        tmp.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# Final save
df["pred_label"]  = pred_label
df["pred_reason"] = pred_reason
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Wrote:", OUTPUT_CSV, "| Rows:", len(df))
df.head(5)


Labeling: 100%|██████████| 258/258 [18:11<00:00,  4.23s/it]

Wrote: /content/drive/MyDrive/llama31_hipaa/predicted_health.csv | Rows: 4127


,text,pred_label,pred_reason
0,English Choose your preferred language  Apple...,0,malformed
1,"In addition to this Privacy Policy, we provide...",0,malformed
2,This product-specific information is accompani...,0,malformed
3,You will be given an opportunity to review thi...,0,malformed
4,You also can view this information at any time...,0,malformed


In [ ]:
!ls /content/drive/MyDrive/llama31_hipaa

ckpt_llama32_1b_hipaa_lora  Health_new_labeled.csv  offload
data			    Health_new_output.csv


In [ ]:
import pandas as pd
OUT = "/content/drive/MyDrive/llama31_hipaa/SmallSamplePredicted.csv"  # update if you saved elsewhere
df = pd.read_csv(OUT)

print("Rows:", len(df))
print("\nLabel counts:")
print(df["pred_label"].value_counts().sort_index())

print("\nSample NON-compliant (0):")
print(df[df.pred_label==0].sample(min(5, (df.pred_label==0).sum()))["text"].tolist())

print("\nSample compliant (1):")
print(df[df.pred_label==1].sample(min(5, (df.pred_label==1).sum()))["text"].tolist())


Rows: 4127

Label counts:
pred_label
0    4127
Name: count, dtype: int64

Sample NON-compliant (0):
['In certain circumstances, the DPF provides the right to invoke binding arbitration to resolve complaints not resolved by other means, as described in Annex I to the DPF Principles .', 'To view cookies on our Site and exercise your rights related to tracking technologies, navigate to the bottom of the page and click on "Your Privacy Choices" or the icon of the cookie.', 'N. Children Our Services are not intended for or directed to children under 18 years of age, and we do not knowingly collect Personal Information from children under the age of', "Information collected includes anonymized usage data of our website, such as specific actions taken on our website, which are tracked and attributed using Meta's pixel.", 'This Payment Policy applies to users who have purchased a HappifyTM subscription plan to access HappifyTM either online or through a mobile application.']

Sample compliant 

In [ ]:
import pandas as pd, json
train = pd.read_json("data/train.jsonl", lines=True)
val   = pd.read_json("data/val.jsonl",   lines=True)
test  = pd.read_json("data/test.jsonl",  lines=True)

def label_from_response(r):
    try:
        return json.loads(r)["label"]
    except Exception:
        return None

train_y = train["response"].map(label_from_response)
val_y   = val["response"].map(label_from_response)
test_y  = test["response"].map(label_from_response)

print("Train counts:", train_y.value_counts().to_dict())
print("Val counts:",   val_y.value_counts().to_dict())
print("Test counts:",  test_y.value_counts().to_dict())

Train counts: {0: 3109, 1: 13}
Val counts: {0: 389, 1: 1}
Test counts: {0: 389, 1: 2}
